# Prior visualisation

**One notebook for every prior variant.** Set `TASK`, run it, and it discovers whichever
pools are on this machine — `original`, `credit_v1`, `credit_v2`, however many you have
generated. Adding a variant needs no edit here.

### Why not one notebook per prior?

Because the interesting question is never *"what does `credit_v1` look like"* on its own.
It is always *"how does `credit_v1` differ from `original`, and from `credit_v2`"*. A
notebook per variant answers the uninteresting question, duplicates all this prose, and
drifts apart the moment one copy gets edited. Worse, it makes the comparison **manual** —
you end up flipping between saved outputs comparing histograms from memory.

So the plots come in two kinds:

| kind | shows | example |
|---|---|---|
| **comparison** | every variant on one axis | boundary mass, base rate, correlation spectrum, the summary table |
| **detail** | one `FOCUS` variant | the 100-histogram grid — you cannot show 100 panels × 4 variants |

### Pools, not a fresh draw

This reads the **actual files training consumed**, so it answers "what did the model
see", not "what would this config produce". If you have no pools it falls back to
generating live, and says so.

All logic lives in `src/visualize/pool_plots.py` and `prior_plots.py`; the shared look is
in `style.py`. This notebook holds none.

**Companion:** `data_exploration.ipynb` shows the real credit datasets these are aimed at.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.visualize import pool_plots as pp, prior_plots, style

style.use_style()
pd.set_option("display.width", 200, "display.max_columns", 40)

# ---- the only two knobs -------------------------------------------------------
TASK  = "lgd"    # "lgd" or "pd" — they measure different things, so one at a time
FOCUS = None     # variant for the DETAIL plots; None = the first non-original one
N     = 100      # datasets to draw per variant
SAVE  = False    # True -> also write PNGs to res/figures/
# -------------------------------------------------------------------------------

style.show_palette();

## 0. What pools are on this machine?

Read the `state` column first:

* **COMPLETE** — the whole pool, exactly what the model trained on.
* **SAMPLE** — a partial download. Fine for every plot here (they use a few hundred
  datasets; one shard is 2,000), but do not quote it as "the pool".

No pools? Either generate them —

```
python scripts/generate_prior.py --config config/LGD.yaml --variant original --all
```

— or copy a sample down from the cluster: `bash scripts/fetch_prior_sample.sh`.
A full pool is ~4 GB (LGD) / ~5.4 GB (PD) **per variant**, so the default fetch takes
one shard per pool instead of all ~19 GB.

In [ ]:
variants = pp.discover_pools(TASK)
print(f"discovered {len(variants)} pool(s) for {TASK.upper()}: {variants or 'none'}\n")
pp.describe_pools(TASK) if variants else print("No pools — section 8 generates live instead.")

## 1. Load every variant

Same seed for each, so the draws are comparable rather than independently lucky.

In [ ]:
loaded = pp.load_all_variants(TASK, n=N, seed=0)
print({k: len(v) for k, v in loaded.items()})

if FOCUS is None:
    non_original = [v for v in loaded if v != "original"]
    FOCUS = non_original[0] if non_original else (list(loaded)[0] if loaded else None)
print(f"FOCUS (detail plots) = {FOCUS}")

## 2. The summary table

**This table is what replaces one-notebook-per-variant.** One row per prior, the columns
that the research question turns on. For LGD that is `in [0,1]` and boundary mass; for PD
it is the base rate.

Expect: `original` near 0 for `in [0,1]`, ours near 1. If they look the same, something is
wrong with the generation, not with the plot.

In [ ]:
summary = pp.variant_summary(loaded, TASK)
summary

## 3. The key comparison

For **LGD**: boundary mass per variant, with the real datasets as stars and dotted lines.
The question to ask is *does any variant's cloud actually cover where the real datasets
sit?* — not merely "is ours different from the original".

For **PD**: base rate per variant, against balance and the real datasets.

Reference values are measured from your processed datasets if they are present, and fall
back to recorded values otherwise (it says which).

In [ ]:
fig = pp.plot_target_comparison(loaded, TASK)

## 4. Target shapes, one row per variant

Ten histograms per variant on one figure. This is the compromise that replaces flipping
between notebooks: not 100 panels for one prior, but enough for each to see the difference
in shape directly.

In [ ]:
fig = pp.plot_target_shapes_by_variant(loaded, n_per=10)

## 5. Correlation spectrum by variant

O'Prior's central measurement — the claim being that what a prior teaches is a
**dependence structure**, not individual functions.

Bold lines are medians. If two variants' spectra sit on top of each other, they teach a
similar structure *however different their targets look* — which would be an important
negative result, not a boring one.

In [ ]:
fig = pp.plot_spectrum_by_variant(loaded)

## 6. Shape sanity check

All variants should look **the same** here. Table shape is not what we are changing, so a
visible difference means an accidental confound, not a finding.

In [ ]:
fig = pp.plot_shapes_by_variant(loaded)

## 7. Detail: one variant up close

Everything below uses `FOCUS` only. These are the plots that cannot be stacked across
variants — change `FOCUS` in the setup cell and re-run this section to look at another.

The functions are the same ones from `prior_plots.py` used on live draws; they work
unchanged on pooled data because a pooled episode is rebuilt into the same
`SyntheticTask` type the generator returns.

In [ ]:
focus_tasks = loaded[FOCUS]
print(f"{FOCUS}: {len(focus_tasks)} datasets")
fig = prior_plots.plot_target_grid(focus_tasks, n_show=min(100, len(focus_tasks)))

In [ ]:
fig = prior_plots.plot_boundary_mass(focus_tasks, real_reference=pp.real_reference(TASK, quiet=True))

In [ ]:
fig = prior_plots.plot_table_shapes(focus_tasks)

In [ ]:
fig = prior_plots.plot_feature_relationships(focus_tasks, n_show=6)

In [ ]:
fig = prior_plots.plot_feature_target_relation(focus_tasks, n_show=8)

## 8. No pools? Generate live instead

The original behaviour, kept as a fallback: draw datasets straight from a config with the
same `TaskGenerator` training uses. Useful for trying a config change *before* spending
CPU hours generating a pool from it.

Note this answers a subtly different question — "what would this config produce" rather
than "what did the model actually see".

In [ ]:
CONFIG = f"config/{TASK.upper()}.yaml"

live_original, info_o = prior_plots.sample_tasks(CONFIG, n=40, credit_fraction=0.0, seed=0)
live_ours, info_c     = prior_plots.sample_tasks(CONFIG, n=40, credit_fraction=1.0, seed=0)
print("original:", info_o["sources"], "| ours:", info_c["sources"])

live = {"original (live)": live_original, "ours (live)": live_ours}
pp.variant_summary(live, TASK)

In [ ]:
fig, comparison = prior_plots.compare_priors(CONFIG, n=40, seed=0)
comparison["ours"]["frac_in_unit"] if "frac_in_unit" in comparison["ours"] else comparison

## 9. Saving figures

`res/figures/` is gitignored — regenerate rather than commit.

In [ ]:
if SAVE:
    print(style.savefig(pp.plot_target_comparison(loaded, TASK), f"res/figures/{TASK}_target_comparison.png"))
    print(style.savefig(pp.plot_target_shapes_by_variant(loaded), f"res/figures/{TASK}_shapes_by_variant.png"))
    print(style.savefig(pp.plot_spectrum_by_variant(loaded), f"res/figures/{TASK}_spectrum_by_variant.png"))
    print(style.savefig(prior_plots.plot_target_grid(focus_tasks), f"res/figures/{TASK}_{FOCUS}_target_grid.png"))
else:
    print("SAVE is False — nothing written. Set SAVE = True in the setup cell.")

## 10. Switching task

`TASK = "pd"` in the setup cell, then Run All. LGD and PD are separate experiments
measuring different quantities, so they are looked at one at a time — but it is the same
notebook, not a second copy.